# The Retriever as a Noisy Channel: Recall, Precision, and Information Limits

This narrative notebook imports the canonical, tested reference implementation
`retriever_as_noisy_channel.py` and walks the topic section by section. The `.py` **owns every
number**; here we just call it so the claims render as executed output.

The previous topic measured how many **bits** a retrieved document adds to the answer,
$I(A;D\mid Q) = H(A\mid Q) - H(A\mid Q,D)$. This topic reads that same machinery as a **communication
channel** and asks what the channel's *recall* and *precision* do to those bits — and what the bits do
to the answer error. Two failure modes, two co-equal headlines: **recall is an erasure**, **precision is
a substitution**.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "retriever-as-noisy-channel"))
import retriever_as_noisy_channel as rnc

c = rnc._corpus()
terms = rnc._clean_channel_terms(c)
print("K (answers/documents) =", c["K"], " N_QUERIES =", c["n_queries"])
print("clean channel: I0 = %.4f bits" % terms["I0"])
print("               H(A|Q)   = %.4f bits" % terms["H_A_given_Q"])
print("               H(A|Q,D) = %.4f bits" % terms["H_cond0"])
print("               Bayes error (clean) = %.4f" % terms["bayes0"])

## Movement 1 — recall is an erasure channel; its capacity is the recall

A retrieval miss **erases** the relevant filing and the generator falls back to a non-informative belief.
Because the erased symbol is independent of the answer given the query, the retriever is a **Binary
Erasure Channel**, and the bits it delivers are *exactly* the recall fraction of the clean bits:
$I_\varepsilon(A;D\mid Q) = \mathrm{recall}\cdot I_0$. The residual entropy rises toward $\log_2 K$.

In [ ]:
rsw = rnc.recall_sweep(c)
print("recall   I=recall*I0   H(A|Q,D)   Bayes   Fano(loose)  Fano(tight)")
for r in rsw["rows"]:
    print("%5.2f    %8.4f    %8.4f   %6.4f   %8.4f     %8.4f" % (
        r["recall"], r["I"], r["H_cond"], r["bayes"], r["fano_loose"], r["fano_tight"]))
rnc.test_bec_capacity_linear()      # I == recall * I0 to machine precision
print("\nBEC identity I = recall*I0 holds to machine precision.")

## Movement 2 — Fano's inequality: the residual entropy is a floor on the answer error

Whatever recall fails to deliver stays behind as residual uncertainty $H(A\mid Q,D)$, and **Fano's
inequality** turns it into a floor on the Bayes error of *any* decoder:
$P_e \ge (H(A\mid Q,D) - 1)/\log_2 K$. The floor is vacuous while $H < 1$ bit, then lifts off and
climbs. At every operating point the model's Bayes error stays above the floor — Fano is a theorem.

In [ ]:
rnc.test_fano_is_a_theorem()        # bayes >= fano_tight >= fano_loose everywhere
rnc.test_recall_degradation_monotone()
rnc.test_fano_activates()
cross = next(r["recall"] for r in rsw["rows"] if r["fano_loose"] > 0)
print("Fano floor first becomes active at recall =", cross,
      "(where H(A|Q,D) crosses 1 bit)")
print("At full erasure (recall 0): H(A|Q,D) =", round(rsw["rows"][-1]["H_cond"], 4),
      "bits, Fano floor =", round(rsw["rows"][-1]["fano_loose"], 4))

## Movement 3 — precision is a substitution channel, and Fano goes blind

A false positive does not erase the relevant filing — it **substitutes** a plausible same-sector
distractor, and the generator reads it and answers *confidently wrong*. The realized error climbs to 1,
yet $H(A\mid Q,D)$ stays **below one bit** (the distractor's posterior is just as sharp, only wrong), so
the Fano floor never moves. The growing realized-vs-Bayes gap is what an entropy bound cannot see.

In [ ]:
ssw = rnc.substitution_sweep(c)
print("eps    realized   Bayes    H(A|Q,D)  Fano   gap")
for r in ssw["rows"]:
    print("%4.1f    %6.4f   %6.4f   %6.4f   %5.3f  %+6.4f" % (
        r["eps"], r["realized"], r["bayes"], r["H_cond"], r["fano_loose"], r["gap"]))
rnc.test_confident_wrong_gap_widens()
print("\nRealized error rises to 1 while the Fano floor stays 0 (H stays < 1 bit): confidently wrong.")

## Movement 4 — the operating point is a rate–distortion choice

Reading more documents raises recall but lowers precision; the generator blends them. On this corpus the
gold filing is rank-1, so recall@k is saturated and contamination shows up not as wrong answers but as
**bits of uncertainty added** — the blended belief drifts toward the full marginal entropy $H(A\mid Q)$.

In [ ]:
print("k   recall@k   precision=1/k   blended-belief entropy (bits)")
for r in rnc.precision_entropy_frontier(c):
    print("%d    %6.3f      %7.4f         %7.4f" % (r["k"], r["recall"], r["precision"], r["H"]))
rnc.test_precision_entropy_frontier()

## Every claim is an assert

Running the full harness re-verifies the always-true anchors (the collapse to the PMI channel, Fano as a
theorem, the entropy identity, the BEC and Fano-floor definitions) and the build-and-run directional
gates (monotone degradation, the floor activating, the confident-wrong gap, the precision frontier), then
prints the constants the laboratory mirrors.

In [ ]:
rnc._run_all()